# dw 教学数仓探查

本地 `docker compose -f docker/docker-compose.yaml` 起的 MySQL 8.0（容器 `mysql`，宿主机端口 `3306`）：
`dw` 是教学数仓，`meta` 是元数据库。账号默认 `early / dili123`（见 `conf/app_config.yaml` 的 `db_dw` 段）。

- 先安装 notebook 依赖：`uv sync --group notebook`，内核选项目的 `.venv`
- 数据覆盖 **2024-01-01 ~ 2025-12-31**，所以“今天”固定取 `2025-12-31`
- 每个 cell 都能单独运行（先运行第 0 节的连接）；改完 SQL 按 `Shift+Enter` 重跑
- 不想跑 Python 生成数据时，可以直接导入 `docker/snapshot/dw_snapshot.sql.gz`，见 `docker/snapshot/README.md`

## 0. 连接

In [ ]:
import os
from pathlib import Path

import pandas as pd
import yaml
from sqlalchemy import create_engine, inspect

# 默认值对齐 conf/app_config.yaml 的 db_dw；要连别处就设环境变量
DW = {
    "host": os.getenv("DW_HOST", "127.0.0.1"),
    "port": int(os.getenv("DW_PORT", "3306")),
    "user": os.getenv("DW_USER", "early"),
    "password": os.getenv("DW_PASSWORD", "dili123"),
    "database": os.getenv("DW_DATABASE", "dw"),
}
engine = create_engine(
    "mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}".format(**DW),
    pool_pre_ping=True,
)
insp = inspect(engine)
# notebook 的工作目录是 notebooks/，项目根目录在上一级
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def q(sql: str, **kwargs) -> pd.DataFrame:
    """执行一条 SQL 并返回 DataFrame；kwargs 作为查询参数传给驱动"""
    with engine.connect() as conn:
        return pd.read_sql(sql, conn, params=kwargs or None)


def cols(table: str) -> str:
    """列出一张表的全部字段名，写 SQL 时速查"""
    return ", ".join(c["name"] for c in insp.get_columns(table))

## 1. 表清单与字段

In [ ]:
q("""
SELECT table_name, table_rows, table_comment
FROM information_schema.tables
WHERE table_schema = 'dw'
ORDER BY table_name
""")

In [ ]:
cols("fact_order")

#### `fact_order` 订单事实表（订单粒度）

| 字段 | 类型 | 说明 |
| --- | --- | --- |
| `order_id` | VARCHAR(30), PK | 订单号 |
| `user_id` | FK → `dim_user` | 下单用户 |
| `shop_id` | FK → `dim_shop` | 店铺 |
| `receiver_region_id` | FK → `dim_region` | **收货城市**（同一张 `dim_region` 被用户常住地和订单收货地两种角色使用） |
| `order_date_id` | FK → `dim_date` | 下单日期 |
| `order_time` | DATETIME | 下单时间 |
| `pay_date_id` | FK → `dim_date`，可空 | 支付日期，未支付为 NULL（**同一张 `dim_date` 有多种日期角色**） |
| `pay_time` | DATETIME，可空 | 支付时间 |
| `platform` | VARCHAR(10) | 下单终端：APP / 小程序 / PC / H5 |
| `payment_method` | VARCHAR(10)，可空 | 支付方式：支付宝 / 微信支付 / 银行卡 / 花呗分期 |
| `order_status` | VARCHAR(10) | 订单状态：待支付 / 已取消 / 已支付 / 已发货 / 已完成（退款单独记在 `fact_refund`，不改订单状态） |
| `item_quantity` | INT | 商品总件数 |
| `total_amount` | DECIMAL | 商品总额 = Σ 明细 `item_amount` |
| `discount_amount` | DECIMAL | 优惠总额 = Σ 明细 `discount_amount` |
| `freight_amount` | DECIMAL | 运费 |
| `order_amount` | DECIMAL | 订单应付金额 = `total_amount − discount_amount + freight_amount`；只有 `pay_time` 不为空时才是实际支付的金额 |

#### `dim_sku` 商品维度

| 字段 | 类型 | 说明 |
| --- | --- | --- |
| `sku_id` | VARCHAR(20), PK | SKU ID |
| `spu_id` | VARCHAR(20) | SPU ID（同一款商品的不同规格共享一个 SPU） |
| `sku_name` | VARCHAR(100) | 商品名称（含规格），如“iPhone 16 Pro 256GB 黑色钛金属” |
| `category_id` | VARCHAR(20), FK → `dim_category` | 三级类目 |
| `brand_name` | VARCHAR(30) | 品牌 |
| `shop_id` | VARCHAR(20), FK → `dim_shop` | 所属店铺 |
| `list_price` | DECIMAL(12,2) | 标价（划线价） |
| `cost_price` | DECIMAL(12,2) | 成本价，用于计算毛利 |
| `launch_date` | DATE | 上架日期 |
| `sku_status` | VARCHAR(10) | 状态：在售 / 下架 |

#### `fact_order_item` 订单明细事实表（订单行粒度）

| 字段 | 类型 | 说明 |
| --- | --- | --- |
| `order_item_id` | VARCHAR(30), PK | 订单明细 ID |
| `order_id` | FK → `fact_order` | 所属订单（下单时间、状态、用户都要通过订单表获取） |
| `sku_id` | FK → `dim_sku` | 商品 |
| `activity_id` | FK → `dim_activity`，可空 | 参与的活动，未参与为 NULL |
| `quantity` | INT | 购买件数 |
| `unit_price` | DECIMAL | 成交单价（可能低于标价） |
| `item_amount` | DECIMAL | 商品金额 = `unit_price × quantity` |
| `discount_amount` | DECIMAL | 分摊到本行的优惠 |
| `pay_amount` | DECIMAL | 本行实付金额 = `item_amount − discount_amount`，不含运费 |

## 2. 指标口径速算与数据自检

`fact_order` 是订单头粒度（一行一个订单），直接聚合不会扇出。

> 坑：订单头金额一旦 join 到 `fact_order_item`（一行一个订单行）再求和，就会被放大。

In [ ]:
q("""
SELECT
  ROUND(SUM(order_amount), 2)                                          AS 下单金额,
  ROUND(SUM(CASE WHEN pay_time IS NOT NULL THEN order_amount END), 2)   AS 支付金额,
  COUNT(DISTINCT CASE WHEN pay_time IS NOT NULL THEN user_id END)        AS 支付买家数,
  ROUND(SUM(CASE WHEN pay_time IS NOT NULL THEN order_amount END)
        / NULLIF(COUNT(DISTINCT CASE WHEN pay_time IS NOT NULL THEN user_id END), 0), 2) AS 客单价
FROM fact_order
""")

In [ ]:
# 订单状态与支付时间一致：待支付 / 已取消没有支付时间，其余状态都有
q("""
SELECT order_status, pay_time IS NOT NULL AS is_paid, COUNT(*) AS orders
FROM fact_order
GROUP BY order_status, is_paid
ORDER BY order_status
""")

In [ ]:
# 订单头与明细对平：两个差异计数都应为 0
q("""
SELECT
  SUM(o.order_amount <> o.total_amount - o.discount_amount + o.freight_amount) AS 订单头公式不符,
  SUM(o.total_amount <> i.item_amount)                                        AS 与明细商品额不符
FROM fact_order o
JOIN (SELECT order_id, SUM(item_amount) AS item_amount
      FROM fact_order_item GROUP BY order_id) i ON i.order_id = o.order_id
""")

## 3. 月度趋势（2024 vs 2025 同月）

销售额按**支付时间**切时间，所以用 `pay_time` 而不是 `order_time`：
下单与支付跨月/跨年的订单，按 `order_time` 分组会被算到下单月去。

In [ ]:
q("""
select 
month(pay_time) as month,
round(sum(case when year(pay_time) = "2024" then order_amount end),2) as y2024,
round(sum(case when year(pay_time) = "2025" then order_amount end),2) as y2025
from 
fact_order 
where pay_time is not null 
group by month(pay_time)
order by month asc
""")

## 4. 类目 × 用户等级交叉

类目走雪花：`fact_order_item → dim_sku → dim_category`；金额用明细的 `pay_amount`（不含运费）。

In [ ]:
q("""
SELECT c.category_l1, u.member_level,
       ROUND(SUM(i.pay_amount), 2) AS sales
FROM fact_order_item i
JOIN fact_order    o ON o.order_id = i.order_id AND o.pay_time IS NOT NULL
JOIN dim_sku       s ON s.sku_id = i.sku_id
JOIN dim_category  c ON c.category_id = s.category_id
JOIN dim_user      u ON u.user_id = o.user_id
GROUP BY c.category_l1, u.member_level
ORDER BY c.category_l1, sales DESC
""")

## 5. 行为 → 支付漏斗

`dim_sku` **没有** `category_l1`（类目在 `dim_category`）。并且要先把两侧各自聚合到“类目”再 join：
行为表和订单表直接 join 会互相放大，`COUNT(DISTINCT o.user_id)` 还会把“在该类目下支付”算成“付过任意商品”。

> `fact_user_behavior` 只有 浏览 / 收藏 / 加购，**没有“下单”**；支付要回到订单表。

In [ ]:
q("""
WITH beh AS (
  SELECT c.category_l1,
         COUNT(DISTINCT CASE WHEN b.behavior_type = '浏览' THEN b.user_id END) AS browse_users,
         COUNT(DISTINCT CASE WHEN b.behavior_type = '加购' THEN b.user_id END) AS cart_users
  FROM fact_user_behavior b
  JOIN dim_sku      s ON s.sku_id = b.sku_id
  JOIN dim_category c ON c.category_id = s.category_id
  GROUP BY c.category_l1
), pay AS (
  SELECT c.category_l1, COUNT(DISTINCT o.user_id) AS pay_users
  FROM fact_order_item i
  JOIN fact_order   o ON o.order_id = i.order_id AND o.pay_time IS NOT NULL
  JOIN dim_sku      s ON s.sku_id = i.sku_id
  JOIN dim_category c ON c.category_id = s.category_id
  GROUP BY c.category_l1
)
SELECT b.category_l1,
       b.browse_users AS 浏览用户,
       b.cart_users   AS 加购用户,
       COALESCE(p.pay_users, 0) AS 支付用户,
       ROUND(COALESCE(p.pay_users, 0) / NULLIF(b.browse_users, 0) * 100, 2) AS 浏览到支付pct
FROM beh b
LEFT JOIN pay p ON p.category_l1 = b.category_l1
ORDER BY b.category_l1
""")

## 6. 退款率（按类目）

分母必须是**该类目全部已支付明细**的金额。原写法从 `fact_refund` 出发 join 明细，
分母只剩“有退款的明细”，算出来会接近 90%。

退款只认 `refund_status = '退款成功'`（`退款中 / 已拒绝 / 已撤销` 都不算）。

In [ ]:
q("""
SELECT c.category_l1,
       ROUND(SUM(COALESCE(r.refund_amount, 0)) / NULLIF(SUM(i.pay_amount), 0) * 100, 2) AS 退款率pct
FROM fact_order_item i
JOIN fact_order   o ON o.order_id = i.order_id AND o.pay_time IS NOT NULL
JOIN dim_sku      s ON s.sku_id = i.sku_id
JOIN dim_category c ON c.category_id = s.category_id
LEFT JOIN fact_refund r
       ON r.order_item_id = i.order_item_id AND r.refund_status = '退款成功'
GROUP BY c.category_l1
ORDER BY 退款率pct DESC
""")

## 7. 核心商品（帕累托）

2025 年每个一级类目里，按销售额从高到低累计，贡献前 80% 销售额的 SKU 算核心商品。

In [ ]:
q("""
WITH sku_sales AS (
  SELECT c.category_l1, i.sku_id, SUM(i.pay_amount) AS sales
  FROM fact_order_item i
  JOIN fact_order   o ON o.order_id = i.order_id
  JOIN dim_sku      s ON s.sku_id = i.sku_id
  JOIN dim_category c ON c.category_id = s.category_id
  WHERE o.pay_time IS NOT NULL
    AND o.pay_date_id BETWEEN 20250101 AND 20251231
  GROUP BY c.category_l1, i.sku_id
), acc AS (
  -- ROWS 窗口逐行累计；默认的 RANGE 会把销售额相同的 SKU 一起累加
  SELECT *,
         SUM(sales) OVER (PARTITION BY category_l1 ORDER BY sales DESC, sku_id
                          ROWS UNBOUNDED PRECEDING) AS acc_sales,
         SUM(sales) OVER (PARTITION BY category_l1) AS cat_sales
  FROM sku_sales
)
SELECT category_l1,
       ROUND(cat_sales, 2) AS 类目销售额,
       COUNT(*) AS sku数,
       SUM(acc_sales - sales < 0.8 * cat_sales) AS 核心sku数,
       ROUND(SUM(CASE WHEN acc_sales - sales < 0.8 * cat_sales THEN sales END)
             / cat_sales * 100, 2) AS 核心销售占比pct
FROM acc
GROUP BY category_l1, cat_sales
ORDER BY cat_sales DESC
""")

## 8. 外键完整性自检

`dw.sql` 没有声明外键约束，关联关系记在 `conf/meta_config.yaml` 各外键字段的 `ref` 里。
这里读出全部关系逐条检查：子表里非空的外键值必须能在父表找到，孤儿行数应全部为 0。

In [ ]:
meta = yaml.safe_load((ROOT / "conf" / "meta_config.yaml").read_text(encoding="utf-8"))
checks = []
for table in meta["tables"]:
    for column in table["columns"]:
        if not column.get("ref"):
            continue
        parent, parent_col = column["ref"].split(".")
        child, child_col = table["name"], column["name"]
        orphans = q(f"""
            SELECT COUNT(*) AS n FROM {child} c
            LEFT JOIN {parent} p ON p.{parent_col} = c.{child_col}
            WHERE c.{child_col} IS NOT NULL AND p.{parent_col} IS NULL
        """)["n"][0]
        checks.append((f"{child}.{child_col}", column["ref"], int(orphans)))
pd.DataFrame(checks, columns=["外键", "关联到", "孤儿行数"])

## 9. 自由查询

下面这条只是模板，改 SQL 直接重跑。

In [ ]:
q("""
SELECT platform,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(SUM(order_amount), 2) AS gmv
FROM fact_order
WHERE pay_time IS NOT NULL
GROUP BY platform
ORDER BY gmv DESC
""")